### 1. Import 

In [39]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_community.document_loaders import PyPDFDirectoryLoader
from dotenv import load_dotenv

In [3]:
load_dotenv()

True

## 2.Create LLM

In [4]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

### 3.Document Loader

In [40]:
from langchain_community.document_loaders import PyPDFLoader


def load_pdf(pdf):
    loader=PyPDFDirectoryLoader("data")
    documents=loader.load()
    return documents

In [36]:
documents=load_pdf("data/PEC_RULES.pdf")

### 4. spliting documents into chunks

In [7]:
def split_documents(documents,chunk_size=1000,chunk_overlap=150):
    splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    return documents

In [8]:
chunks=split_documents(documents)

In [9]:
len(chunks)

196

### 5.Embedding Model

In [10]:
from langchain_huggingface import  HuggingFaceEmbeddings

def load_embedding_model():
    embedding_model=HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )
    return embedding_model


In [11]:
embedding_model = load_embedding_model()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [12]:
embedding_model

HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

### 6.Vector Store


In [13]:
def create_vector_store(chunks, embedding_model):
    vector_store = FAISS.from_documents(
        chunks,
        embedding_model
    )
    return vector_store

In [14]:
embedding_model = load_embedding_model()

vector_store = create_vector_store(
    chunks,
    embedding_model
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

### 7.storing vector 

In [15]:
def save_vector_store(vector_store,folder_path="faiss_index"):
    vector_store.save_local(folder_path)

### 8.Load Vector Store

In [16]:
def load_vector_store(folder_path,embedding_model):
    vector_store=FAISS.load_local(
        folder_path,
        embedding_model,
        allow_dangerous_deserialization=True
    )
    return vector_store

In [17]:
loaded_vector_store = load_vector_store(
    "faiss_index",
    embedding_model
)

### 9.create retriver

In [18]:
def create_retriever(vector_store, k=10):
    retriever = vector_store.as_retriever(
        search_kwargs={"k": k}
    )
    return retriever

In [19]:
retriever = create_retriever(vector_store)

In [20]:
print(vector_store)

### 10.create prompt

In [21]:
def create_prompt():

    prompt = ChatPromptTemplate.from_template("""
You are an AI assistant.

Answer the user's question using ONLY the provided context.

Context:
{context}

Question:
{question}
""")

    return prompt

In [22]:
prompt=create_prompt()

In [23]:
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="\nYou are an AI assistant.\n\nAnswer the user's question using ONLY the provided context.\n\nContext:\n{context}\n\nQuestion:\n{question}\n"), additional_kwargs={})])

## 10.Format Documents

In [24]:
def format_docs(docs):

    return "\n\n".join(
        doc.page_content for doc in docs
    )

# create rag chain

In [25]:
def create_rag_chain(retriever, prompt, llm):

    rag_chain = (
        {
            "context": retriever | format_docs,
            "question": RunnablePassthrough(),
        }
        | prompt
        | llm
        | StrOutputParser()
    )

    return rag_chain

                    load_pdf()
                         │
                         ▼
                   documents
                         │
                         ▼
                split_documents()
                         │
                         ▼
                      chunks
                         │
         ┌───────────────┴──────────────┐
         ▼                              ▼
load_embedding_model()          embedding_model
         │                              │
         └───────────────┬──────────────┘
                         ▼
             create_vector_store()
                        │
                         ▼
                  vector_store
                         │
                         ▼
               create_retriever()
                         │
                         ▼
                    retriever

create_prompt() ───────────────► prompt

ChatGoogleGenerativeAI() ──────► llm

retriever + prompt + llm
            │
            ▼
    create_rag_chain()
            │
            ▼
        rag_chain

In [26]:
embedding_model = load_embedding_model()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [45]:
vector_store = load_vector_store(
    "faiss_index",
    embedding_model
)

In [28]:
retriever = create_retriever(vector_store)

In [29]:
prompt = create_prompt()

In [34]:
load_dotenv()

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    google_api_key="GOOGLE_API_KEY"
)

In [31]:
rag_chain = create_rag_chain(
    retriever,
    prompt,
    llm
)

In [32]:
response = rag_chain.invoke(
    "how many departments are there in prathyusha engineering college"
)

print(response)

Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 39.329300206s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 20
}
, retry_del

The provided context does not explicitly state the total number of departments in Prathyusha Engineering College. It mentions that ECE, EEE, IT, and Mechanical departments achieved 100% placement, and that Anna University Research Recognition is for "all departments."


In [ ]:
response=rag_chain.invoke(
    "give the details about class rooms and about strenght"
)
print(response)

The following details are available about classrooms:

**CLASS ROOMS**
*   **AI-DS BLOCK:** 16 rooms, 90.00 sq.m. area, 60 capacity
*   **CSE BLOCK:** 15 rooms, 90.00 sq.m. area, 60 capacity
*   **ECE - EEE - MECH BLOCK:** 11 rooms, 90.00 sq.m. area, 60 capacity
*   **IT-AIML-CYBSEC BLOCK:** 20 rooms, 90.00 sq.m. area, 60 capacity
*   **MAIN BLOCK:** 23 rooms, 90.00 sq.m. area, 60 capacity

All classrooms are in Permanent blocks.


In [35]:
documents = load_pdf("data/PEC_RULES.pdf")

chunks = split_documents(documents)

embedding_model = load_embedding_model()

vector_store = create_vector_store(
    chunks,
    embedding_model
)

save_vector_store(vector_store)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [41]:
documents = load_pdf("data")

print(f"Total pages loaded: {len(documents)}")

Total pages loaded: 209


In [43]:
question = input("Ask a question: ")

response = rag_chain.invoke({
    "question": question,
    "chat_history": ""
})

print("\nAnswer:\n")
print(response)

AttributeError: 'dict' object has no attribute 'replace'